# Phase 6 — Why the cross-dataset test failed: feature distributions of CICIDS2017 vs CIC-IoT2023
* Data: the 13 unified features of the merged configuration (features_v3), in their original units
  (the merged Min–Max scaling and log transform are undone, as in P5).
* Only the TRAINING split is used, so the test splits stay untouched.
* For each feature, overall and within each class shared by both datasets: median, 25th/75th percentile,
  share of zeros, and the two-sample Kolmogorov–Smirnov statistic D between the two datasets
  (univariate KS tests for detecting dataset shift: Rabanser et al., 2019). D = 0 means identical
  distributions, D = 1 means no overlap at all.
* Runs on a normal CPU runtime (no GPU needed). Outputs: outputs_v3/feature_shift/ (CSV tables and figures).

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — load the merged training split in original units
import os, json, time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FEAT = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/config1_merged/'
OUT  = '/content/drive/MyDrive/Thesis/outputs_v3/feature_shift/'
os.makedirs(OUT, exist_ok=True)
CLASS_NAMES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
SRC = ['CICIDS2017', 'CIC-IoT2023']
SEED, MAX_ROWS = 42, 200_000            # rows sampled per dataset and class (all rows when fewer)

def log(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)

sp = json.load(open(FEAT + 'spec.json'))
feats = sp['features']
X = np.load(FEAT + 'X_train.npy').astype(np.float64)
y = np.load(FEAT + 'y_train.npy').astype(np.int64)
s = np.load(FEAT + 'source_train.npy')
mn, mx = np.array(sp['train_min']), np.array(sp['train_max'])
use_log = np.array([f in sp['log_features'] for f in feats])
lo = np.where(use_log, np.log1p(np.maximum(mn, 0)), mn)
hi = np.where(use_log, np.log1p(np.maximum(mx, 0)), mx)
rng = np.where(hi > lo, hi - lo, 1.0)
X = X * rng + lo
X[:, use_log] = np.expm1(X[:, use_log])
log(f'training split: {len(y):,} rows, {len(feats)} features | ' +
    ', '.join(f'{k}: {(s == k).sum():,}' for k in SRC))

rs = np.random.default_rng(SEED)
def sample(mask):
    idx = np.flatnonzero(mask)
    return X[rs.choice(idx, MAX_ROWS, replace=False)] if len(idx) > MAX_ROWS else X[idx]

groups = {('ALL', k): sample(s == k) for k in SRC}
shared = [c for c in range(len(CLASS_NAMES)) if all(((s == k) & (y == c)).any() for k in SRC)]
for c in shared:
    for k in SRC:
        groups[(CLASS_NAMES[c], k)] = sample((s == k) & (y == c))
log('classes in both datasets: ' + ', '.join(CLASS_NAMES[c] for c in shared))
del X; import gc; gc.collect()

In [ ]:
# %% Cell 2 — statistics per feature and class
rows = []
for g in ['ALL'] + [CLASS_NAMES[c] for c in shared]:
    a, b = groups[(g, SRC[0])], groups[(g, SRC[1])]
    for j, f in enumerate(feats):
        d = stats.ks_2samp(a[:, j], b[:, j]).statistic
        r = dict(group=g, feature=f, ks_D=round(float(d), 3))
        for k, v in ((SRC[0], a[:, j]), (SRC[1], b[:, j])):
            q25, q50, q75 = np.percentile(v, [25, 50, 75])
            r.update({f'{k} median': q50, f'{k} p25': q25, f'{k} p75': q75, f'{k} zeros %': round(100 * float((v == 0).mean()), 1)})
        rows.append(r)
tab = pd.DataFrame(rows)
tab.to_csv(OUT + 'feature_shift_stats.csv', index=False)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20)
print('\nKS statistic D between CICIDS2017 and CIC-IoT2023 (0 = same distribution, 1 = no overlap):')
piv = tab.pivot(index='feature', columns='group', values='ks_D')[['ALL'] + [CLASS_NAMES[c] for c in shared]]
piv['mean over classes'] = piv[[CLASS_NAMES[c] for c in shared]].mean(1)
piv = piv.sort_values('mean over classes', ascending=False)
print(piv.round(3).to_string())
piv.to_csv(OUT + 'ks_by_feature_and_class.csv')

for g in ['ALL', 'BENIGN', 'DDoS']:
    print(f'\nMedians (p25 – p75) for {g}:')
    t = tab[tab.group == g].set_index('feature')
    print(pd.DataFrame({k: t[f'{k} median'].map('{:.4g}'.format) + '  (' + t[f'{k} p25'].map('{:.3g}'.format) + ' – ' +
                        t[f'{k} p75'].map('{:.3g}'.format) + ')' for k in SRC} | {'KS D': t['ks_D']}).to_string())

In [ ]:
# %% Cell 3 — figures: distribution of each feature in both datasets (BENIGN and DDoS)
for g in ['BENIGN', 'DDoS']:
    fig, axes = plt.subplots(3, 5, figsize=(18, 10)); axes = axes.ravel()
    for j, f in enumerate(feats):
        ax = axes[j]
        for k, col in zip(SRC, ['#2b6cb0', '#c05621']):
            v = groups[(g, k)][:, j]
            v = np.log10(1 + np.maximum(v, 0))
            ax.hist(v, bins=60, alpha=0.55, density=True, color=col, label=k)
        d = float(tab[(tab.group == g) & (tab.feature == f)].ks_D.iloc[0])
        ax.set_title(f'{f}  (KS D = {d:.2f})', fontsize=10); ax.set_xlabel('log10(1 + value)', fontsize=8)
    for ax in axes[len(feats):]:
        ax.axis('off')
    axes[0].legend(fontsize=8)
    fig.suptitle(f'{g} records: feature distributions in CICIDS2017 and CIC-IoT2023 (training split)', fontsize=13)
    fig.tight_layout(); fig.savefig(OUT + f'feature_shift_{g}.png', dpi=150); plt.close(fig)
    log(f'saved feature_shift_{g}.png')
log('done: tables and figures saved to ' + OUT)